# Regression investment score sintetis

Membandingkan ElasticNet dan Random Forest dengan weighted baseline. Target formula sintetis hanya untuk pengujian pipeline dan kontrak artefak.

Semua input dan output saat ini adalah prototipe sintetis. Hasil tidak boleh dianggap sebagai observasi lapangan atau rekomendasi bisnis/investasi produksi.

In [1]:
from pathlib import Path

def find_root():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / 'ml' / 'DATASET_CATALOG.md').exists():
            return candidate
    raise FileNotFoundError('Jalankan notebook dari dalam repository TCI')

ROOT = find_root()
ML_ROOT = ROOT / 'ml'
SEED = 20260911
STATUS = 'synthetic_prototype'
print(f'Project root: {ROOT}')

Project root: C:\Users\axels\Axel Documents\Documents\BINUS\Lomba\MAPID WebGIS (Top 50)\App\TCI


In [2]:

import json
import pickle
import platform
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import sklearn
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import ElasticNet
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

feature_dir = ML_ROOT / "regression_scoring"
data_path = feature_dir / "data" / "station_month_features.csv"
models_dir, outputs_dir = feature_dir / "models", feature_dir / "outputs"
models_dir.mkdir(parents=True, exist_ok=True)
outputs_dir.mkdir(parents=True, exist_ok=True)
df = pd.read_csv(data_path)
df["period_date"] = pd.to_datetime(df.period + "-01")

features = ["passenger_volume_monthly", "passenger_growth_yoy_pct", "business_density_750m", "poi_density_750m", "pedestrian_access_index", "land_property_availability_index", "intermodal_connectivity_index", "estimated_rental_index_rp_thousand_m2_month", "distance_to_cbd_km"]
target = "investment_potential_score"
duplicates = int(df.duplicated(["station_code", "period"]).sum())
missing = {column: int(value) for column, value in df[features + [target]].isna().sum().items() if value}
out_of_range_targets = int((~df[target].between(0, 100)).sum())
invalid_status = int((df.source_status != STATUS).sum())
quality_passed = not (duplicates or missing or out_of_range_targets or invalid_status)
quality_report = {"passed": quality_passed, "record_count": len(df), "station_count": int(df.station_code.nunique()), "period_min": df.period.min(), "period_max": df.period.max(), "duplicate_station_periods": duplicates, "missing_values": missing, "targets_outside_0_100": out_of_range_targets, "invalid_source_status": invalid_status}
(outputs_dir / "data_quality_report.json").write_text(json.dumps(quality_report, indent=2), encoding="utf-8")
if not quality_passed:
    raise ValueError("Regression quality gate failed; outputs are unavailable")

train = df[df.period_date < "2025-01-01"].copy()
test = df[df.period_date >= "2025-01-01"].copy()
X_train, y_train = train[features], train[target]
X_test, y_test = test[features], test[target]

def weighted_score(frame, reference):
    definitions = {
        "passenger_volume_monthly": (0.26, True),
        "passenger_growth_yoy_pct": (0.12, True),
        "pedestrian_access_index": (0.17, True),
        "business_density_750m": (0.14, True),
        "land_property_availability_index": (0.15, True),
        "intermodal_connectivity_index": (0.16, True),
    }
    total = np.zeros(len(frame), dtype=float)
    breakdown = {}
    for column, (weight, positive) in definitions.items():
        low, high = reference[column].min(), reference[column].max()
        normalized = ((frame[column] - low) / (high - low)).clip(0, 1) if high > low else pd.Series(0.5, index=frame.index)
        if not positive:
            normalized = 1 - normalized
        breakdown[column] = normalized * weight * 100
        total += breakdown[column].to_numpy()
    return total, pd.DataFrame(breakdown, index=frame.index)

elastic = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()), ("model", ElasticNet(alpha=0.05, l1_ratio=0.2, random_state=SEED, max_iter=20000))])
forest = Pipeline([("imputer", SimpleImputer(strategy="median")), ("model", RandomForestRegressor(n_estimators=400, min_samples_leaf=3, max_features=0.8, random_state=SEED, n_jobs=-1))])
candidates = {"elastic_net": elastic, "random_forest": forest}
candidate_metrics = {}
for name, pipeline in candidates.items():
    pipeline.fit(X_train, y_train)
    prediction = np.clip(pipeline.predict(X_test), 0, 100)
    candidate_metrics[name] = {"mae": mean_absolute_error(y_test, prediction), "rmse": mean_squared_error(y_test, prediction) ** 0.5}
weighted_prediction, _ = weighted_score(X_test, X_train)
candidate_metrics["weighted_prototype_baseline"] = {"mae": mean_absolute_error(y_test, weighted_prediction), "rmse": mean_squared_error(y_test, weighted_prediction) ** 0.5}
selected_name = min(("elastic_net", "random_forest"), key=lambda name: candidate_metrics[name]["mae"])
selected_pipeline = candidates[selected_name]

# Refit the selected ML pipeline on all synthetic development records for export.
selected_pipeline.fit(df[features], df[target])
latest = df.sort_values("period_date").groupby("station_code", as_index=False).tail(1).copy()
weighted_latest, breakdown = weighted_score(latest[features], df[features])
latest["model_predicted_investment_score"] = np.clip(selected_pipeline.predict(latest[features]), 0, 100).round(2)
latest["investment_score"] = weighted_latest.round(2)
latest["rank"] = latest.investment_score.rank(method="min", ascending=False).astype(int)
latest["investment_priority"] = pd.cut(latest.investment_score, bins=[-np.inf, 45, 70, np.inf], labels=["Low", "Medium", "High"], right=False).astype(str)
latest["score_method"] = "weighted_prototype"
latest["model_method"] = f"{selected_name}_synthetic_prototype"
latest["source_status"] = STATUS
score_columns = ["station_code", "station_id", "station_name", "line", "period", "investment_score", "rank", "investment_priority", "score_method", "model_predicted_investment_score", "model_method", "source_status"]
latest[score_columns].sort_values("rank").to_csv(outputs_dir / "station_scores.csv", index=False)

breakdown.insert(0, "station_name", latest.station_name.to_numpy())
breakdown.insert(0, "station_id", latest.station_id.to_numpy())
breakdown.insert(0, "station_code", latest.station_code.to_numpy())
breakdown["investment_score"] = weighted_latest.round(2)
breakdown["score_method"] = latest.score_method.to_numpy()
breakdown["source_status"] = STATUS
breakdown.to_csv(outputs_dir / "indicator_breakdown.csv", index=False)

if selected_name == "random_forest":
    importance = dict(zip(features, selected_pipeline.named_steps["model"].feature_importances_))
else:
    importance = dict(zip(features, np.abs(selected_pipeline.named_steps["model"].coef_)))
importance = dict(sorted(((key, float(value)) for key, value in importance.items()), key=lambda item: item[1], reverse=True))
artifact = {"pipeline": selected_pipeline, "features": features, "selected_model": selected_name, "target": target, "source_status": STATUS, "warning": "Synthetic target; not a production investment score."}
with (models_dir / "investment_score_pipeline.pkl").open("wb") as file:
    pickle.dump(artifact, file)
metrics = {"selected_model": selected_name, "selection_metric": "holdout MAE", "published_score_method": "weighted_prototype", "split": {"train": "2023-01..2024-12", "test": "2025-01..2025-12"}, "candidates": candidate_metrics, "feature_importance_not_causal": importance, "source_status": STATUS, "limitation": "Targets are formula-generated; metrics only measure reproduction of synthetic patterns. The published score is deterministic so its indicator breakdown is exact."}
(outputs_dir / "metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
(outputs_dir / "feature_schema.json").write_text(json.dumps({"features": features, "target": target, "identifier_fields": ["station_code", "station_id"], "excluded_from_training": ["record_id", "station_name", "line", "period", "investment_priority", "target_provenance", "source_status"]}, indent=2), encoding="utf-8")
manifest = {"run_at_utc": datetime.now(timezone.utc).isoformat(), "source": str(data_path.relative_to(ROOT)), "records": len(df), "period_range": [df.period.min(), df.period.max()], "model": selected_name, "seed": SEED, "python": platform.python_version(), "sklearn": sklearn.__version__, "source_status": STATUS, "limitations": "Synthetic proxies and formula-generated target; API must not present this as a production predictive score."}
(outputs_dir / "run_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(json.dumps({"quality_passed": quality_passed, "selected_model": selected_name, "test_metrics": candidate_metrics[selected_name], "model": str(models_dir / 'investment_score_pipeline.pkl')}, indent=2))


{
  "quality_passed": true,
  "selected_model": "random_forest",
  "test_metrics": {
    "mae": 3.0803164811434858,
    "rmse": 3.90200722664229
  },
  "model": "C:\\Users\\axels\\Axel Documents\\Documents\\BINUS\\Lomba\\MAPID WebGIS (Top 50)\\App\\TCI\\ml\\regression_scoring\\models\\investment_score_pipeline.pkl"
}
